# 0. מהדאטה אל שורת המודל

קריטריון ההצלחה: כל האובייקטים נבנו, עברו בדיקות בריאות, והערכים החסרים ממופים עם הסבר.

**דאטה מאומת → team_matches → שורת פיצ׳רים → מודל → מטריצת תוצאות 9×9**

אנחנו מכינים את כלי המחקר וממחישים את הקלט למודל; אין כאן אימון. עונות ההחזקה והעונה החיה נשארות נעולות.

## 1. סביבת העבודה
נשתמש בסביבת הפרויקט ונאפשר רענון של פונקציות בזמן חקירה. היקף הניתוח סגור לעונות 2006/07–2023/24.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

from ligat.analysis.prepare import (
    ANALYSIS_SEASONS,
    COLUMN_GROUPS,
    check_analysis_tables,
    describe_missing_values,
    flag_unusual_matches,
    load_analysis_tables,
    outcome_base_rates,
    preview_pre_match_row,
    team_history,
)

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 25)
pio.renderers.default = "plotly_mimetype+notebook"
colors = {"בית": "#2563eb", "תיקו": "#6b7280", "חוץ": "#f59e0b"}
print(f"היקף: 2006/07–2023/24; {len(ANALYSIS_SEASONS)} עונות. ללא עונות 26–28.")

## 2. טעינת שש התצוגות
כל טבלה מתארת יחידה אחרת — משחק, קבוצה במשחק, שער, קבוצה בעונה, עונה או קדנציית מאמן. הטעינה מסוננת מראש ואינה כותבת לדאטה הנקי.

In [ ]:
tables = load_analysis_tables()
units = ["משחק", "קבוצה במשחק", "שער", "קבוצה בעונה", "עונה", "קדנציית מאמן"]
inventory = pd.DataFrame(
    [
        {
            "טבלה": name,
            "שורה =": unit,
            "מפתח": ", ".join(frame.index.names),
            "שורות": len(frame),
            "עמודות": len(frame.columns),
        }
        for (name, frame), unit in zip(vars(tables).items(), units, strict=True)
    ]
)
display(inventory)

## 3. משחק עם הקשר
נראה תוצאה, מחצית ואדומים ליד הקבוצות והתאריך. השערים באים ממשחקים, ולא מטבלאות הליגה ההיסטוריות.

In [ ]:
match_columns = [
    "date",
    "home_team_name",
    "away_team_name",
    "home_goals",
    "away_goals",
    "ht_home_goals",
    "ht_away_goals",
    "home_red_cards",
    "away_red_cards",
]
display(tables.match_table[match_columns].head(8))

## 4. איזה מידע מותר לפני המשחק?
מזהים מתארים את המשחק; pre_match ידוע מראש; in_match קרה במשחק. החלוקה מונעת שימוש בתוצאה כפיצ׳ר של אותו משחק.

In [ ]:
column_groups = pd.DataFrame(
    [
        {"קבוצה": name, "מספר עמודות": len(columns), "דוגמאות": ", ".join(columns[:3])}
        for name, columns in COLUMN_GROUPS.items()
    ]
)
display(column_groups)

נציג בנפרד את המידע שהיה זמין לפני השריקה. נתוני מאמנים ושווי עשויים להיות חסרים — החסר נשמר ולא מוחלף בניחוש.

In [ ]:
display(tables.match_table[COLUMN_GROUPS["pre_match"]].head(5))

## 5. אותה תוצאה, שתי נקודות מבט
ניצחון בית הוא הפסד חוץ: team_matches נותנת לכל קבוצה שורה משלה, עם שערים ונקודות מנקודת מבטה.

In [ ]:
team_columns = [
    "date",
    "team_name",
    "opponent_name",
    "is_home",
    "goals_scored",
    "goals_conceded",
    "result",
    "points_earned",
    "ht_state",
]
display(tables.team_matches[team_columns].head(8))

נבדוק על משחק אחד שכל צד מופיע פעם אחת. שתי השורות מונעות טיפול שונה ומבלבל בקבוצת בית ובקבוצת חוץ.

In [ ]:
example_game = tables.match_table.index[0]
display(tables.team_matches.xs(example_game, level="game_id")[team_columns])

## 6. ההיסטוריה של מכבי חיפה
שליפה לפי השם הקנוני מחזירה את כל משחקי הקבוצה בסדר כרונולוגי. נסתכל בעשרת המשחקים האחרונים בתקופת הניתוח.

In [ ]:
haifa = team_history(tables.team_matches, "מכבי חיפה")
display(haifa[team_columns].tail(10))

השאלה הכדורגלנית: איך הצטברו הנקודות על המגרש בעונת 2022/23? זו ספירה ממשחקים שנשמרו, ללא הורדות מנהלתיות.

In [ ]:
haifa_season = haifa.loc[haifa.season_id.eq(24)].copy()
haifa_season["נקודות מצטברות"] = haifa_season.points_earned.cumsum()
fig = px.line(
    haifa_season,
    x="date",
    y="נקודות מצטברות",
    title=f"מכבי חיפה צברה {int(haifa_season.points_earned.sum())} נקודות על המגרש ב־2022/23",
    labels={"date": "תאריך"},
    color_discrete_sequence=["#2563eb"],
)
fig.update_layout(template="plotly_white", yaxis_title="נקודות", yaxis_rangemode="tozero")
fig.show()

## 7. שערים כאירועים
בכל שער רשומה הקבוצה שנהנתה ממנו, גם בשער עצמי. half מבדיל תוספת זמן במחצית הראשונה מדקה דומה במחצית השנייה.

In [ ]:
display(tables.goal_events.head(10))

השאלה: באילו חלקי המשחק מובקעים השערים? ערכים עם טווח דקה לא ידוע אינם מומצאים ומוצגים בנפרד מהגרף.

In [ ]:
minute_counts = (
    tables.goal_events.minute_bucket.value_counts(sort=False).rename("שערים").reset_index()
)
display(pd.DataFrame({"טווח לא ידוע": [int(tables.goal_events.minute_bucket.isna().sum())]}))
largest_bucket = minute_counts.loc[minute_counts["שערים"].idxmax(), "minute_bucket"]
fig = px.bar(
    minute_counts,
    x="minute_bucket",
    y="שערים",
    title=f"הטווח שבו הובקעו הכי הרבה שערים: {largest_bucket}",
    labels={"minute_bucket": "טווח דקות"},
    color_discrete_sequence=["#6b7280"],
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero")
fig.show()

## 8. קבוצה בעונה
נשווה את 2022/23 לפי המיקום הסופי, נקודות על המגרש ושווי. נתח השווי הנוכחי הוא תיאור בלבד; למודל מותר נתח העונה הקודמת.

In [ ]:
season_teams = tables.team_seasons.xs(24, level="season_id").sort_values("final_position")
display(
    season_teams[
        [
            "team_name",
            "final_position",
            "points_per_match",
            "goals_scored",
            "goals_conceded",
            "total_value_eur",
            "value_share_of_league",
            "prev_season_value_share",
        ]
    ].head(10)
)

## 9. תמונה של 18 עונות
הסיכומים מאפשרים להשוות עידנים בלי קיזוז והורדות נקודות. כאן בלבד נציג את כל 18 השורות, שהן טבלה קטנה.

In [ ]:
display(tables.season_summary)

השאלה: כמה משחקים מסתיימים בניצחון בית, תיקו או ניצחון חוץ בכל עונה? אותם צבעים מייצגים את התוצאות בכל המחברת.

In [ ]:
home_min = tables.season_summary.home_win_rate.min()
home_max = tables.season_summary.home_win_rate.max()
result_rates = tables.season_summary[["home_win_rate", "draw_rate", "away_win_rate"]].rename(
    columns={"home_win_rate": "בית", "draw_rate": "תיקו", "away_win_rate": "חוץ"}
)
result_rates.index = [f"{s + 1998}/{str(s + 1999)[-2:]}" for s in result_rates.index]
result_rates = (
    result_rates.mul(100)
    .rename_axis("עונה")
    .reset_index()
    .melt(id_vars="עונה", var_name="תוצאה", value_name="% מהמשחקים")
)
fig = px.bar(
    result_rates,
    x="עונה",
    y="% מהמשחקים",
    color="תוצאה",
    color_discrete_map=colors,
    title=f"שיעור ניצחונות הבית נע בין {home_min:.0%} ל־{home_max:.0%}",
)
fig.update_layout(template="plotly_white", barmode="stack", yaxis_range=[0, 100])
fig.show()

## 10. מאמנים וכיסוי המקור
קדנציה היא רצף נצפה, לא תאריך מינוי מוכח. שני הדגלים מבדילים התחלת כיסוי מאוחרת וריצוד קצר A→B→A מהחלפת מאמן ודאית.

In [ ]:
display(tables.coach_spells.head(10))

נספור את דגלי האיכות לפי עונת תחילת הקדנציה. אין תיקון אוטומטי של המאמנים או מחיקת רצפים; הדגלים מזהירים את החוקר.

In [ ]:
coach_quality = tables.coach_spells.groupby("season_id_start")[
    ["is_left_censored", "is_flicker"]
].sum()
coach_quality.index = [f"{s + 1998}/{str(s + 1999)[-2:]}" for s in coach_quality.index]
display(coach_quality)

## 11. בדיקות הבריאות
הבדיקות מכשילות חוסר שלמות או אי־התאמה. אזהרות מוכרות נשארות גלויות — עבר אינו אומר שאין חוסרים.

In [ ]:
warnings = check_analysis_tables(tables)
passed_checks = [
    "שתי שורות לכל משחק",
    "נקודות ושערים על המגרש",
    "קבוצה פעם במחזור",
    "מספר משחקים לפי פורמט והחרגות",
    "זוגות בסיבוב הסדיר",
    "שערים ומחצית מול תוצאה",
    "מספר קבוצות, יורדות וכיסוי שווי",
    "קדנציות ללא חפיפה",
]
display(pd.DataFrame({"בדיקה": passed_checks, "מצב": "עבר"}))

נציג את כל האזהרות המתועדות ולא רק מספר כללי. הסיבות מסבירות אילו ניתוחים ידרשו סינון או תשומת לב.

In [ ]:
display(pd.DataFrame({"מצב": "אזהרה מתועדת", "סיבה": warnings}))

## 12. נפח המשחקים בזמן
השאלה: האם יש הפסקות בלוח המשחקים? חודשי אפס מוצגים גם הם, כדי שהפסקת הקורונה לא תיעלם מהתרשים.

In [ ]:
months = tables.match_table.date.dt.tz_localize(None).dt.to_period("M")
monthly = (
    months.value_counts()
    .sort_index()
    .reindex(pd.period_range(months.min(), months.max(), freq="M"), fill_value=0)
)
monthly_plot = pd.DataFrame({"חודש": monthly.index.to_timestamp(), "משחקים": monthly.values})
fig = px.line(
    monthly_plot,
    x="חודש",
    y="משחקים",
    title=f"באפריל 2020 שוחקו {int(monthly.loc['2020-04'])} משחקי ליגה",
    color_discrete_sequence=["#6b7280"],
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero", yaxis_title="משחקים בחודש")
fig.show()

## 13. מפת החוסרים
סף warn הוא מעל 5% ו־fail מעל 30%. fail עם הסבר מוכר הוא אזהרה; בלי הסבר הוא חוסם את הניתוח.

In [ ]:
missing = describe_missing_values(tables)
display(
    missing.loc[
        missing.status.ne("ok"), ["table", "column", "n_missing", "share", "status", "reason"]
    ]
)

השאלה: אילו עמודות דורשות הכי הרבה זהירות? added_time חסרה תמיד כי המקור רושם דקות רצות; אין צורך להשלים אותה.

In [ ]:
leading_missing = missing.loc[missing.n_missing.gt(0)].nlargest(12, "share").copy()
leading_missing["עמודה"] = leading_missing["table"] + "." + leading_missing.column
leading_missing["% חסר"] = leading_missing.share * 100
fig = px.bar(
    leading_missing.sort_values("% חסר"),
    x="% חסר",
    y="עמודה",
    orientation="h",
    title="added_time חסרה ב־100% מהשערים: המקור משתמש בדקות רצות",
    color_discrete_sequence=["#6b7280"],
)
fig.update_layout(template="plotly_white", xaxis_range=[0, 100])
fig.show()

## 14. משחקים חריגים
חריגים מסומנים ולא נמחקים. משחק עם הרבה שערים או אדומים עשוי להיות אמיתי, וחשוב לדעת מה נכלל לפני פרשנות סטטיסטית.

In [ ]:
unusual = flag_unusual_matches(tables)
flags = ["seven_or_more_goals", "three_or_more_red_cards", "is_postponed", "borrowed_venue"]
display(unusual[flags].sum().rename("משחקים").to_frame())

נראה דוגמאות לחריגים עם שמות ותאריכים, כדי שאפשר יהיה לחזור להקשר הכדורגלני. מספרי הסוגים עשויים לחפוף.

In [ ]:
display(unusual.head(10))

## 15. מהדאטה למודל
נבחר משחק מכבי חיפה–בית״ר בעונת 2022/23. הפיצ׳רים משתמשים רק בעשרת המשחקים שקדמו לזמן המשחק; זה אב־טיפוס, לא מנגנון הפיצ׳רים הסופי.

In [ ]:
selected = tables.match_table.loc[
    tables.match_table.season_id.eq(24)
    & tables.match_table.home_team_id.eq(1005)
    & tables.match_table.away_team_id.eq(3595)
]
selected_game_id = int(selected.index[0])
display(selected[match_columns].head(1))

זה מה שהמודל יראה: ממוצעי עבר ונתח שווי מהליגה של העונה הקודמת. מצב מחצית חסר או סותר לא מושלם; ממוצע מחצית נשען רק על ערכים תקינים.

In [ ]:
preview = preview_pre_match_row(tables, selected_game_id)
display(preview.filter(like="feature_").T.rename(columns={selected_game_id: "ערך פיצ׳ר"}))

זה מה שהמודל ינסה לחזות: תוצאת המשחק. הפלט הסופי המתוכנן הוא מטריצת 9×9 של תוצאות 0:0–8:8, הסתברויות בית/תיקו/חוץ ושערים צפויים לכל קבוצה.

In [ ]:
display(preview.filter(like="target_").T.rename(columns={selected_game_id: "יעד בפועל"}))

## 16. בנצ׳מרק B0
שיעורי התוצאות הם הרף הפשוט הראשון להשוואה. כאן הם תיאור כל התקופה; בבקטסט מחשבים אותם על האימון בלבד. רווח Wilson הוא רווח סמך נפרד לכל שיעור, לא רווח משותף.

In [ ]:
base_rates = outcome_base_rates(tables.match_table)
display(base_rates.loc[base_rates.scope.eq("overall")])

נציג גם את שיעורי הבסיס לפי עונה. שינוי בין עונות הוא סיבה לבדוק יציבות בבקטסט כרונולוגי, ולא לערבב עונות באקראי.

In [ ]:
display(
    base_rates.loc[base_rates.scope.eq("season")].pivot(
        index="season_id", columns="outcome", values="rate"
    )
)

השאלה: מהי התוצאה השכיחה בתקופה הזו ומה אי־הוודאות בשיעורה? פסי השגיאה מראים רווח סמך Wilson של 95%.

In [ ]:
overall = base_rates.loc[base_rates.scope.eq("overall")].copy()
overall["תוצאה"] = overall.outcome.map({"home_win": "בית", "draw": "תיקו", "away_win": "חוץ"})
overall["% מהמשחקים"] = overall.rate * 100
overall["שגיאה עליונה"] = (overall.ci_upper - overall.rate).clip(lower=0) * 100
overall["שגיאה תחתונה"] = (overall.rate - overall.ci_lower).clip(lower=0) * 100
winner = overall.loc[overall.rate.idxmax()]
fig = px.bar(
    overall,
    x="תוצאה",
    y="% מהמשחקים",
    color="תוצאה",
    color_discrete_map=colors,
    error_y="שגיאה עליונה",
    error_y_minus="שגיאה תחתונה",
    title=f"{winner['תוצאה']} היא התוצאה השכיחה: {winner.rate:.1%} מהמשחקים",
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero")
fig.show()

## 17. מה למדנו, ומה נשאל במחברת 02?

- כל המשחקים מופיעים פעמיים בנקודת מבט קבוצה; נקודות על המגרש מאפשרות להשוות עידנים.
- דקות רצות מחייבות שימוש ב־half: דקה 47 יכולה להיות תוספת זמן במחצית הראשונה.
- added_time חסרה במקור; מאמנים בתחילת הכיסוי וריצוד דורשים זהירות בפרשנות החלפות.
- שווי נוכחי מתאר בלבד; שווי t−1 לעולות נשאר חסר ואין השלמה בשלב הזה.

**למחברת 02:** האם יתרון הבית יציב בין עידנים? כיצד שווי יחסי t−1 קשור לתוצאות? האם מצב מחצית מסביר ניהול משחק אחרי סינון הסתירות? האם החלפות מאמנים נראות אחרת כשמחריגים ריצוד וכיסוי מאוחר?

אלה שאלות להמשך, לא החלטות מודל או מסקנות סיבתיות.